In [3]:
# Using PyGod built in DOMINANT architecture for my data

In [4]:
from pygod.detector import DOMINANT
from torch_geometric.data import Data, Batch
import pandas as pd
import numpy as np
import torch

In [ ]:
def get_active_stocks(returns, t, lookback_days, feature_dfs=None, min_obs=21, eps=0.0):
    t = pd.to_datetime(t)
    window = returns.loc[t - pd.Timedelta(days=lookback_days): t]

    # enough non-NaN observations
    counts = window.notna().sum(axis=0)
    ok_obs = counts >= min_obs # check how many NaN values in the window for each stock, if less than min_obs then False, else True
    #print(f'Observation counts for time {t}: {counts} \nStocks with enough observations: {window.columns[ok_obs].tolist()}')

    # not constant zero in the window (treat as missing asset)
    if eps == 0.0:
        ok_nonzero = ~(window.fillna(0.0) == 0.0).all(axis=0)
    else:
        ok_nonzero = ~(window.fillna(0.0).abs() <= eps).all(axis=0)
    #print(f'Stocks with non-zero returns for time {t}: {window.columns[ok_nonzero].tolist()}') WORKING
    #print(f'this is for ok_nonzero: {window.columns[ok_nonzero].tolist()}')
    #print(f'this is for ok_obs: {window.columns[ok_obs].tolist()}') PROBLEM this is empty
    active = window.columns[ok_nonzero].tolist() # finds columns that satisfy both conditions size is (number of stocks,)
    #print(f'Active stocks length: {len(active)}')
    if feature_dfs is not None:
        active_set = set(active)
        #print('active set' , active_set)
        for df in feature_dfs:
            # only care if the stock exists as a column in the dataframe
            if not df.empty:
                # Find intersection between current active stocks and this dataframe's columns
                active_set = active_set.intersection(df.columns)
        
        active = list(active_set)

    return active

In [7]:
def prepare_node_features(stocks, sectors, volatility, market_caps, pe_ratios, implied_vol, short_interest,
                           beta, operating_margin, return_on_equity, rsi_momentum, turnover, t):

    rows = []
    t = pd.to_datetime(t)
    if not stocks:
        return torch.empty((0, 2), dtype=torch.float32)
    for stock in stocks:
        # Get factors for stock at time t (or default values if missing)
        sector_id = sectors.loc[stock, 'sector_id'] if stock in sectors.index else 0
        market_cap = market_caps.loc[t, stock] if t in market_caps.index and stock in market_caps.columns else 0.0
        pe_ratio = pe_ratios.loc[t, stock] if t in pe_ratios.index and stock in pe_ratios.columns else 0.0
        implied_volatility = implied_vol.loc[t, stock] if t in implied_vol.index and stock in implied_vol.columns else 0.0
        short_int = short_interest.loc[t, stock] if t in short_interest.index and stock in short_interest.columns else 0.0
        beta_val = beta.loc[t, stock] if t in beta.index and stock in beta.columns else 0.0
        op_margin = operating_margin.loc[t, stock] if t in operating_margin.index and stock in operating_margin.columns else 0.0
        roe = return_on_equity.loc[t, stock] if t in return_on_equity.index and stock in return_on_equity.columns else 0.0
        rsi = rsi_momentum.loc[t, stock] if t in rsi_momentum.index and stock in rsi_momentum.columns else 0.0
        turn = turnover.loc[t, stock] if t in turnover.index and stock in turnover.columns else 0.0

        # Get volatility at time t (or nearest available)
        if t in volatility.index and stock in volatility.columns:
            vol = volatility.loc[t, stock]
        else:
            # Get closest date
            available_dates = volatility.index[volatility.index <= t]
            if len(available_dates) > 0:
                closest_date = available_dates[-1]
                vol = volatility.loc[closest_date, stock]
            else:
                vol = 0.0  # Default if no data available
        rows.append([sector_id, vol, market_cap, pe_ratio, implied_volatility, short_int, beta_val, op_margin, roe, rsi, turn])
    # features is (N, 11)
    features = np.array(rows, dtype=np.float32)
    features = np.nan_to_num(features, nan=0.0, posinf=0.0, neginf=0.0)#


    if len(features) >0:
        num_cols = features.shape[1]
        for i in range(num_cols):  # Normalize each feature to [0, 1]
            if features[:, i].max() > features[:, i].min():
                features[:, i] = (features[:, i] - features[:, i].min()) / (features[:, i].max() - features[:, i].min() + 1e-8)

    non_zero_count = np.count_nonzero(features)
    total_elements = features.size
    zero_fraction = 1.0 - (non_zero_count / total_elements)
    
    if zero_fraction > 0.9: # If more than 90% of data is zero
        print(f"\n[WARNING] Time {t}: {zero_fraction*100:.1f}% of features are ZERO.")
        print("Sample Row (first stock):", features[0])
        # Check raw dataframe lookup for one stock to debug
        test_stock = stocks[0]
        print(f"Debug check for {test_stock} at {t}:")
        if test_stock in pe_ratios.columns:
            # Check if exact date exists
            date_exists = t in pe_ratios.index
            print(f"  - Date {t} in PE_ratios index? {date_exists}")
            if not date_exists:
                # Show nearest dates
                print(f"  - PE_ratios nearby dates: {pe_ratios.index[pe_ratios.index.get_indexer([t], method='nearest')]}")
                
    return torch.tensor(features, dtype=torch.float32)


def create_edge_index(corr_matrix, k_neighbors=15):
    """
    Creates an edge index where each node connects to its top-k most correlated peers.
    """
    N = corr_matrix.shape[0]
    if isinstance(corr_matrix, np.ndarray):
        corr = torch.tensor(corr_matrix, dtype=torch.float32)
    else:
        corr = corr_matrix.clone()

    #diagonal to -infinity so a node doesn't select itself
    mask_diag = torch.eye(N, dtype=torch.bool, device=corr.device)
    corr.masked_fill_(mask_diag, float('-inf'))
    
    # includes strong positive and strong negative correlations
    vals, indices = torch.topk(corr.abs(), k=min(k_neighbors, N-1), dim=1)
    
    # create Edge List
    
    src_list = torch.arange(N, device=corr.device).repeat_interleave(k_neighbors)
    trg_list = indices.flatten()
    
    edge_index = torch.stack([src_list, trg_list], dim=0)
    
    return edge_index

In [8]:
def correlation_matrix(returns, t, K, eps=0.0, active=None):
    t = pd.to_datetime(t)
    windowed_returns = returns.loc[t - pd.Timedelta(days=K*1.5): t].dropna(how='all')
    window = windowed_returns.dropna(axis=1, how="all")
    if active is not None:
        window = window[active]

    if eps == 0.0:
        window = window.loc[:, ~(window.fillna(0.0) == 0.0).all(axis=0)]
    else:
        window = window.loc[:, ~(window.fillna(0.0).abs() <= eps).all(axis=0)]
    active_cols = window.columns.tolist()

    #scaler = MinMaxScaler(feature_range=(0, 1))
    #windowed_returns = scaler.fit_transform(windowed_returns)
    #windowed_returns = (windowed_returns - windowed_returns.min()) / (windowed_returns.max() - windowed_returns.min() + 1e-8)
    corr_matrix = window.corr().values
    corr_matrix = np.nan_to_num(corr_matrix, nan=0.0, posinf=0.0, neginf=0.0)

    return corr_matrix, active_cols

In [ ]:
# load all data
def load_and_fix_index(filename):
    df = pd.read_csv(filename, index_col=0)
    df.index = pd.to_datetime(df.index, format='%m/%d/%Y') # Fix date format
    df = df[~df.index.duplicated(keep='last')]
    df.dropna(how='all', inplace=True)
    df = df.ffill().bfill()
    return df

# Load Constituent Factors (Tables where Cols = Tickers, Rows = Dates)
Market_caps = load_and_fix_index('Data/SPX_Constituents_market_cap_2006_2025(in).csv')

PE_ratios = load_and_fix_index('Data/SPX_Constituents_Calculated_PE_2006_2025(in).csv')

Implied_vol = load_and_fix_index('Data/SPX_Constituents_Implied_vol_2006_2025(in).csv')

Beta = load_and_fix_index('Data/SPX_Constituents_Beta_2006_2025(in).csv')
Operating_margin = load_and_fix_index('Data/SPX_Constituents_Op_Margin_2006_2025(in).csv')   
Return_on_equity = load_and_fix_index('Data/SPX_Constituents_Ret_On_Equity_2006_2025(in).csv')
RSI_momentum = load_and_fix_index('Data/SPX_Constituents_RSI_momentum_2006_2025(in).csv')
Short_interest = load_and_fix_index('Data/SPX_Constituents_Short_Interest_Pct_2006_2025(in).csv')
Turnover = load_and_fix_index('Data/SPX_Constituents_Turnover_30D_2006_2025(in).csv')
sectors = pd.read_excel('SPX_sectors_data.xlsx', sheet_name='Sectors', 
                        header=0, index_col=0)
sectors['sector_id'] = sectors['Sector'].astype('category').cat.codes
# Example placeholder setup to make this runnable in context
# Replace these lines with your actual data loading block
# ---------------------------------------------------------
returns = pd.read_excel('SPX_sectors_data.xlsx', header=[0,1], index_col=0)
returns.columns = returns.columns.get_level_values(0)
returns.dropna(how='all', inplace=True) 
returns = returns.pct_change().dropna(how='all')
returns = returns.ffill().bfill()
all_stocks = returns.columns.get_level_values(0).unique().tolist()
# ---------------------------------------------------------
train_returns = returns.loc['2012-01-01':'2019-12-31']
test_returns = returns.loc['2020-01-01':'2024-12-31']
volatility = returns.rolling(window=21).std().dropna(how='all') * np.sqrt(252)
feature_dfs_list = [
    Market_caps, PE_ratios, Implied_vol, Beta, 
    Operating_margin, Return_on_equity, RSI_momentum, 
    Short_interest, Turnover, volatility
]
K = 21
# Train/Test Split
train_dates = train_returns.index # Example subset
test_dates = test_returns.index
prices = pd.read_excel('SPX_sectors_data.xlsx', header=[0,1], index_col=0)
prices.dropna(how='all', inplace=True)
prices = prices.ffill().bfill()
prices.columns = prices.columns.droplevel(1)
test_prices = prices.loc['2020-01-01':'2024-01-31']
train_prices = prices.loc['2012-01-01':'2019-12-31']
train_volatility = volatility.loc['2012-01-01':'2019-12-31']

In [33]:

K = 21
# 1. Prepare a list of static graphs from your time series
data_list = []

for t in train_returns.index:
    stocks = get_active_stocks(train_returns, t, lookback_days=K, feature_dfs=feature_dfs_list)

    if len(stocks) <= 15:
        continue
    
    # Use your existing logic to get features and edges for time t
    x = prepare_node_features(stocks, sectors, volatility, Market_caps, PE_ratios, Implied_vol, Short_interest,
                               Beta, Operating_margin, Return_on_equity, RSI_momentum, Turnover, t)
    
    corr_matrix, _ = correlation_matrix(train_returns, t, K, eps=0.0, active=stocks)

    edge_index = create_edge_index(corr_matrix, k_neighbors=15)

    # Create static PyG Data object
    snapshot = Data(x=x, edge_index=edge_index)
    data_list.append(snapshot)

print(f"Total snapshots collected: {len(data_list)}")
# 2. Combine into a single large batch
# This allows the model to see many 'versions' of a normal graph
train_data = Batch.from_data_list(data_list)

# 3. Initialize and Fit
detector = DOMINANT(hid_dim=32, epoch=100)
detector.fit(train_data)

# 4. Predict on a specific test day
test_snapshot = test_returns.index
scores = detector.decision_function(test_snapshot)

Active stocks length: 495
Active stocks length: 497
Active stocks length: 497
Active stocks length: 497
Active stocks length: 497
Active stocks length: 497
Active stocks length: 497
Active stocks length: 497
Active stocks length: 497
Active stocks length: 497
Active stocks length: 497
Active stocks length: 497
Active stocks length: 497
Active stocks length: 497
Active stocks length: 497
Active stocks length: 497
Active stocks length: 497
Active stocks length: 497
Active stocks length: 497
Active stocks length: 497
Active stocks length: 497
Active stocks length: 497
Active stocks length: 497
Active stocks length: 497
Active stocks length: 497
Active stocks length: 497
Active stocks length: 497
Active stocks length: 497
Active stocks length: 497
Active stocks length: 497
Active stocks length: 497
Active stocks length: 497
Active stocks length: 497
Active stocks length: 497
Active stocks length: 497
Active stocks length: 497
Active stocks length: 497
Active stocks length: 497
Active stock

RuntimeError: [enforce fail at alloc_cpu.cpp:117] data. DefaultCPUAllocator: not enough memory: you tried to allocate 3936954398976 bytes.